# Hyperview — Hydra Runner
Mount Drive, install dependencies, then run  with any config override.
All experiment config is in  — **do not edit ** to change hyperparameters.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, subprocess

# ---------- paths ----------
PROJECT_DIR  = "/content/drive/MyDrive/hyperview"   # project root on Drive
TRAIN_TAR    = "/content/drive/MyDrive/train.tar.gz"
TEST_TAR     = "/content/drive/MyDrive/test.tar.gz"
TRAIN_GT     = "/content/drive/MyDrive/train_gt.csv"
WAVELENGTHS  = "/content/drive/MyDrive/wavelengths.csv"

# ---------- extract data ----------
os.makedirs("/content/train_data", exist_ok=True)
os.makedirs("/content/test_data",  exist_ok=True)
!tar -xzf {TRAIN_TAR} -C /content/train_data
!tar -xzf {TEST_TAR}  -C /content/test_data
!cp {TRAIN_GT}    /content/train_gt.csv
!cp {WAVELENGTHS} /content/wavelengths.csv

# ---------- change working dir to project ----------
os.chdir(PROJECT_DIR)
print("Working dir:", os.getcwd())

In [ ]:
# Install dependencies (run once per Colab session)
!pip install -q hydra-core omegaconf timm wandb torchgeo joblib tqdm

In [ ]:
# ── W&B API key setup ──────────────────────────────────────────────
# Option A (recommended): store key in Colab Secrets
#   Runtime → Manage secrets → add  WANDB_API_KEY  with your key
import os
try:
    from google.colab import userdata
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    print("W&B API key loaded from Colab Secrets.")
except Exception:
    # Option B: paste key directly (never commit this to git)
    # os.environ["WANDB_API_KEY"] = "paste-your-key-here"
    print("WANDB_API_KEY not found in Colab Secrets."
          " wandb.login() will prompt interactively, or set it above.")

# Option C: pass key at runtime (no env var needed):
#   python train.py wandb.api_key=your_key_here


## Run training
Use Hydra CLI overrides to switch experiments **without editing any file**.



In [ ]:
# ---- Edit the override string to run any experiment ----
overrides = "model=convnext_tiny data.pca_components=10"

!python train.py {overrides}

## Submission inference
Run after training finishes.

In [ ]:
# Reuse the saved artefacts from training
import torch, joblib, numpy as np, pandas as pd
from torch.utils.data import DataLoader
from src.dataset import load_data, transform_patches_with_mask, NPZDataset
from src.model   import HyperspectralRegressor

OUTPUT_DIR = "/content/outputs"

scaler_x  = joblib.load(f"{OUTPUT_DIR}/scaler_hyperspectral.pkl")
pca       = joblib.load(f"{OUTPUT_DIR}/pca_hyperspectral.pkl")
scaler_y  = joblib.load(f"{OUTPUT_DIR}/scaler_labels.pkl")

n_components = pca.n_components_

# --- load test patches ---
X_test_sub = load_data("/content/test_data/test")
X_test_pca = transform_patches_with_mask(X_test_sub, scaler_x, pca)

# --- dataset (no labels, no augmentation) ---
test_sub_ds     = NPZDataset(X_test_pca, augment=False)
test_sub_loader = DataLoader(test_sub_ds, batch_size=64, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# NOTE: backbone_name must match what was trained
model = HyperspectralRegressor(
    in_channels=n_components,
    backbone_name="convnext_tiny_in22k",   # <-- change to match your run
    pretrained=False,
).to(device)
model.load_state_dict(torch.load(f"{OUTPUT_DIR}/best_model.pth", map_location=device))
model.eval()

preds_scaled = []
with torch.no_grad():
    for X_b, nv_b in test_sub_loader:
        preds_scaled.append(model(X_b.to(device), nv_b.to(device)).cpu().numpy())

y_pred = scaler_y.inverse_transform(np.vstack(preds_scaled))
pd.DataFrame(y_pred, columns=["P","K","Mg","pH"]).to_csv("submission.csv", index_label="sample_index")
print("submission.csv saved")